In [1]:
import os
from cerebras.cloud.sdk import Cerebras
import pandas as pd
import numpy as np
import json
from config_gen import generate_coalitional_conf, generate_divergent_conf, generate_minorty_conf, generate_uniform_conf
from strats import ADD, MPL, APP, LMS, MAJ, FAI,MAJ_from_df, BORDA, AWM, BORDA_from_df
import random
import time
import re
from ollama import Client
from cerebras.cloud.sdk import RateLimitError
import ast
from scipy.stats import kendalltau, spearmanr
from itertools import combinations


In [ ]:
client_ol = Client() #requires ollama logged in on device (mac). Pipeline, however, is adaptable for e.g, Huggingface. Only the LLM call needs to be switched.


### Helper functions which are needed further down


def dcg_at_k(relevance_scores, k=10):
    relevance_scores = np.array(relevance_scores)[:k]
    return np.sum(relevance_scores / np.log2(np.arange(2, len(relevance_scores) + 2)))

def ndcg_at_k(predicted_order, gold_order, k=10, binary_relevance=False):

    if binary_relevance:
        gold_set = set(gold_order)
        predicted_relevance = [1 if item in gold_set else 0 for item in predicted_order]
        ideal_relevance = [1] * len(gold_order)  
    else:
        relevance_map = {item: len(gold_order) - i for i, item in enumerate(gold_order)}
        predicted_relevance = [relevance_map.get(item, 0) for item in predicted_order]
        ideal_relevance = [relevance_map[item] for item in gold_order]
    
    dcg = dcg_at_k(predicted_relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg



In [ ]:
## Initialization of group generation (configurations) and social choice-based aggregation strategies

random.seed(time.time())

num_items = 50
group_size = 4
domains = ['tourist locations', 'movies', 'anon']

configuration_list = ['divergent', 'uniform', 'coalitional', 'minority']
configurations = {
    "coalitional": lambda: generate_coalitional_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "divergent":   lambda: generate_divergent_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "minority":    lambda: generate_minorty_conf(n=group_size, m=num_items, r=100, options=ITEMS),
    "uniform":     lambda: generate_uniform_conf(n=group_size, m=num_items, r=100, options=ITEMS),
}

strategies = {
    "ADD": lambda df: ADD(df),
    "MAJ": lambda df: MAJ_from_df(df),
    "LMS": lambda df: LMS(df),
    "MPL": lambda df: MPL(df),
    "APP": lambda df: APP(df, threshold=60),
    #"FAI": lambda: FAI(result),
    "BORDA": lambda df: BORDA_from_df(df),
    "AWM": lambda df: AWM(df, threshold=35),
}

In [ ]:

preprocessed_dataset_folder = "datasets/movielens_dataset"
threshold = 0.75 ## popularity threshold (percentile) to only include most popular movie titles/tourist destinations

m_ratings = pd.read_csv(preprocessed_dataset_folder + "/ratings.csv")
m_titles = pd.read_csv(preprocessed_dataset_folder + "/movies.csv")

rating_counts = (
    m_ratings
    .groupby('item')
    .size()
    .rename('rating_count')
)

rating_cutoff = rating_counts.quantile(threshold) ## In the current study, we only use the most popular movies.

selected_items = rating_counts[
    rating_counts >= rating_cutoff
].index.tolist()

selected_ratings_df = m_ratings[m_ratings['item'].isin(selected_items)]
m_included = m_titles[m_titles['item'].isin(selected_items)]

titles = m_included['title'].tolist()

print(f"{len(m_included)} movie titles included | threshold n ratings: {rating_cutoff:.0f}")

tourist = pd.read_csv('datasets/Tourist_Destinations.csv')
tourist.columns = [
    "Destination Name","Country","Continent","Type",
    "Avg Cost (USD/day)","Best Season","Avg Rating",
    "visits","UNESCO Site"
]

visit_cutoff = tourist['visits'].quantile(threshold)

tourist = tourist[tourist['visits'] >= visit_cutoff]

tourist['Loc'] = tourist['Destination Name'] + ', ' + tourist['Country']
locations = tourist['Loc'].tolist()

print(f"{len(locations)} locations included | threshold visits: {visit_cutoff:.2f}")


In [ ]:
import os
import json
import re
import time
import pandas as pd

file_path = 'results-stability.csv'
if os.path.exists(file_path):
    df_results = pd.read_csv(file_path)

    completed = set(
        zip(
            df_results['groupID'].astype(str),
            df_results['llm'],
            df_results['run_id'].astype(int)
        )
    )
else:
    columns = ['groupID','configuration', 'group_size','num_items','domain',
               'llm', 'recommendation', 'explanation',
               'ADD', 'APP', 'LMS', 'MPL', 'MAJ', 'group', 'run_id']
    df_results = pd.DataFrame(columns=columns)
    completed = set()

all_groups = pd.read_csv('datasets/groups.csv')
dfs = all_groups.sample(n=25, random_state=42).to_dict('records')

llms = ["ministral-3:8b-cloud", "gpt-oss:20b-cloud", "gpt-oss:120b-cloud", "mistralai/mistral-large-2512"]

errorcount = 0
ers = []
###############

def parse_llm_json(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        cleaned = re.sub(r"```(?:json)?", "", text).strip()
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in LLM output")
        return json.loads(match.group())

for group_obj in dfs:
    try:
        groupid = str(group_obj['groupID'])
        num_items = group_obj['num_items']
        member_ratings = group_obj['group'] 
        domain = 'anon' 
        
        for llm_name in llms:
            for run_id in range(1, 6):
                
                if (groupid, llm_name, run_id) in completed:
                    continue

                system_message = {
                    'role': 'system',
                    'content': f"""
You are tasked with making group recommendations based on the different preferences of the group members.
You need explain the process behind making the recommendation to the group in such a way that someone without recommender systems knowledge can understand.
The information you are provided contain the preferences of the group. Every candidate item for recommendation has a rating from each user listed in the order by user (first rating from user1, second from user2 etc).
The rating is a scale from 0 to 10. For the recommendation, you simply mention the item identifier.
You make a recommendation to the group of users by providing a ranking of the items based on the recommendation approach you came up with.

Provide your answer as VALID JSON ONLY.
Do not use markdown or code fences.
Do not include newlines inside string values.
Use plain ASCII characters only.

Format:
{{
"recommendation": ["item1","item2","item3","item4","item5","item6","item7","item8","item9","item10"],
"explanation": "Short explanation of how you made the recommendation with no line breaks"
}}
"""
                }
                
                scenario = {
                    'role': 'user',
                    'content': f"""
The per-item ratings are presented below:
### BEGIN TABLE ###
{member_ratings}
### END TABLE ###

Think about the answer internally, but only output the final JSON object (containing recommendation ranking and explanation). Do not include any additional text or python code.
Return STRICT JSON. Do not use markdown.
"""
                }
                
                messages = [system_message, scenario]
                
                if 'gpt' in llm_name.lower() or 'stral' in llm_name.lower(): 
                    resp = client_ol.chat(llm_name, messages=messages, options={"temperature": 0.5})
                    out = resp['message']['content']
      
                else:
                    resp = client.chat.completions.create(
                        messages=messages,
                        model=llm_name,
                        stream=False,
                        max_completion_tokens=1000,
                        temperature=0.5,
                    )
                    out = resp.choices[0].message.content

                if "<think>" in out or "</think>" in out:
                    out = re.sub(r'^.*?</think>', '', out, flags=re.DOTALL).strip()
                    
                out = parse_llm_json(out)
                recommendation = out['recommendation']
                explanation = out['explanation']

                new_row = pd.DataFrame([{
                    'groupID': groupid,
                    'num_items': num_items,
                    'domain': domain,
                    'llm': llm_name,
                    'recommendation': str(recommendation), 
                    'explanation': explanation,
                    'group': str(member_ratings),
                    'run_id': run_id
                }])

                if df_results.empty:
                    df_results = new_row
                else:
                    df_results = pd.concat([df_results, new_row], ignore_index=True)

                df_results.to_csv(file_path, index=False)
                completed.add((groupid, llm_name, run_id))
                time.sleep(2)

    except Exception as e: 
        errorcount += 1
        ers.append(str(e))
        print(f"Error encountered: {e}")
        if errorcount > 5:
            break
        if "high traffic" in str(e).lower() or "queue_exceeded" in str(e).lower() or "limit" in str(e).lower():
            time.sleep(20)
            continue

In [2]:


df = pd.read_csv('results-stability.csv')
df['recommendation'] = df['recommendation'].apply(lambda x: ast.literal_eval(x)[:10])

def calculate_metrics(group_runs):
    """
    Calculates mean pairwise Kendall's Tau, Spearman's Rho, 
    and Jaccard similarity for a list of ranked recommendation lists.
    """
    rankings = group_runs['recommendation'].tolist()
    
    if len(rankings) < 2:
        return pd.Series({'tau': np.nan, 'spearman': np.nan, 'jaccard': np.nan})

    tau_scores = []
    spearman_scores = []
    jaccard_scores = []
    
    for r1, r2 in combinations(rankings, 2):
        tau, _ = kendalltau(r1, r2)
        tau_scores.append(tau)
        

        rank_map = {item: i for i, item in enumerate(r1)}
        r1_mapped = [i for i in range(len(r1))]
        r2_mapped = [rank_map.get(item, len(r1)) for item in r2]
        
        if len(set(r2_mapped)) <= 1:
            rho = 0.0
        else:
            rho, _ = spearmanr(r1_mapped, r2_mapped)
        spearman_scores.append(rho)
        
        set1, set2 = set(r1), set(r2)
        if len(set1.union(set2)) == 0:
            jaccard = 1.0 
        else:
            jaccard = len(set1.intersection(set2)) / len(set1.union(set2))
        jaccard_scores.append(jaccard)
        
    return pd.Series({
        'tau': np.mean(tau_scores), 
        'spearman': np.mean(spearman_scores),
        'jaccard': np.mean(jaccard_scores)
    })

group_stability = df.groupby(['llm', 'groupID']).apply(calculate_metrics, include_groups=False)

model_stability = group_stability.groupby('llm').mean()


print(model_stability.round(3).to_markdown())


| llm                          |   tau |   spearman |   jaccard |
|:-----------------------------|------:|-----------:|----------:|
| gpt-oss:120b-cloud           | 0.942 |      0.994 |     0.975 |
| gpt-oss:20b-cloud            | 0.754 |      0.944 |     0.876 |
| ministral-3:8b-cloud         | 0.171 |      0.777 |     0.701 |
| mistralai/mistral-large-2512 | 0.356 |      0.901 |     0.819 |
